# Shortest Paths and String Matching

```{contents}
:local:
:depth: 2
```

Algorithm assumptions determine what an answer means. BFS minimizes hops, Dijkstra minimizes nonnegative total weights, and a string matcher needs a precise equality and occurrence policy. Reusing justified information can eliminate repeated work without changing the contract.

## Minimum Hops Is Not Minimum Cost

```{index} Minimum Hops Is Not Minimum Cost
```

Suppose HQ→Client has one edge costing 100, while HQ→Helpdesk→Client costs 1+1. BFS chooses one hop; a nonnegative weighted shortest-path algorithm chooses cost two. Chapter 23 provides the complete guarded Dijkstra implementation. Negative weights require another algorithm, and a reachable negative cycle on a route to the target can mean no finite minimum exists.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var graph=new GraphWalk(new Dictionary<string,string[]>{["HQ"]=new[]{"Client","Helpdesk"},["Helpdesk"]=new[]{"Client"},["Client"]=Array.Empty<string>()});
var bfs=graph.Bfs("HQ");
Console.WriteLine($"minimum-hop route={string.Join("->",bfs.PathTo("Client"))}, hops={bfs.Distance["Client"]}");
Console.WriteLine("that route costs 100; feasible two-hop route costs 2");
public sealed class GraphWalk
{
    private readonly Dictionary<string,string[]> graph;
    private readonly string[] vertices;
    public GraphWalk(Dictionary<string,string[]> adjacency)
    {
        ArgumentNullException.ThrowIfNull(adjacency);
        graph=new Dictionary<string,string[]>(StringComparer.Ordinal);
        foreach(var pair in adjacency)
        {
            if(pair.Value is null)throw new ArgumentException("Null adjacency list.");
            graph.Add(pair.Key,pair.Value.Distinct(StringComparer.Ordinal).ToArray());
        }
        foreach(var neighbors in graph.Values)
            if(neighbors.Any(v=>v is null||!graph.ContainsKey(v)))throw new ArgumentException("Unknown endpoint.");
        vertices=graph.Keys.OrderBy(v=>v,StringComparer.Ordinal).ToArray();
    }
    public sealed class Layers
    {
        public string Source { get; }
        public string[] Order { get; }
        public IReadOnlyDictionary<string,int?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        public long EdgesExamined { get; }
        internal Layers(string source,string[] order,Dictionary<string,int?> distance,Dictionary<string,string?> previous,long edges)
        {
            Source=source;Order=order;EdgesExamined=edges;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,int?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            while(at is not null)
            {
                if(path.Count>=Distance.Count)throw new InvalidOperationException("Predecessor cycle.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Invalid source path.");
            return path.ToArray();
        }
    }
    public Layers Bfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var distance=vertices.ToDictionary(v=>v,_=>(int?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new Queue<string>();var order=new List<string>();long edges=0;
        distance[source]=0;queue.Enqueue(source);
        while(queue.Count>0)
        {
            string at=queue.Dequeue();order.Add(at);trace?.Invoke($"visit {at}: layer={distance[at]}");
            foreach(string next in graph[at])
            {
                edges++;
                if(distance[next] is not null)continue;
                distance[next]=distance[at]!.Value+1;previous[next]=at;queue.Enqueue(next);
            }
        }
        return new Layers(source,order.ToArray(),distance,previous,edges);
    }
    public record Depth(string[] Preorder,string[] FinishOrder,long EdgesExamined);
    public Depth Dfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var color=vertices.ToDictionary(v=>v,_=>0,StringComparer.Ordinal);
        var stack=new Stack<(string Vertex,int Next)>();var pre=new List<string>();var finish=new List<string>();long edges=0;
        Enter(source);
        while(stack.Count>0)
        {
            var frame=stack.Pop();
            if(frame.Next==graph[frame.Vertex].Length)
            {color[frame.Vertex]=2;finish.Add(frame.Vertex);trace?.Invoke($"exit {frame.Vertex}");continue;}
            stack.Push((frame.Vertex,frame.Next+1));string next=graph[frame.Vertex][frame.Next];edges++;
            if(color[next]==0)Enter(next);
            else if(color[next]==1)trace?.Invoke($"active edge {frame.Vertex}->{next}");
        }
        return new Depth(pre.ToArray(),finish.ToArray(),edges);
        void Enter(string vertex){color[vertex]=1;pre.Add(vertex);stack.Push((vertex,0));trace?.Invoke($"enter {vertex}");}
    }
    public string[][] UndirectedComponents()
    {
        var neighbors=graph.ToDictionary(p=>p.Key,p=>p.Value.ToHashSet(StringComparer.Ordinal),StringComparer.Ordinal);
        foreach(var pair in graph)
            foreach(string next in pair.Value)
                if(!neighbors[next].Contains(pair.Key))throw new ArgumentException("Components require symmetric adjacency.");
        var seen=new HashSet<string>(StringComparer.Ordinal);var groups=new List<string[]>();
        foreach(string root in vertices)
        {
            if(!seen.Add(root))continue;
            var queue=new Queue<string>();var group=new List<string>();queue.Enqueue(root);
            while(queue.Count>0)
            {
                string at=queue.Dequeue();group.Add(at);
                foreach(string next in graph[at])if(seen.Add(next))queue.Enqueue(next);
            }
            groups.Add(group.ToArray());
        }
        return groups.ToArray();
    }
}


This is a verified objective counterexample, not a reason to call BFS incorrect. Declare units, direction, permitted weights, and whether the requested result is reachability, edge count, total cost, or all destinations before selecting an algorithm.

## Direct Matching with an Explicit Contract

```{index} Direct Matching with an Explicit Contract
```

Search exact ordinal UTF-16 code units, returning every start position, including overlapping matches. A missing nonempty pattern returns an empty array. An empty pattern matches each boundary 0..text.Length; null arguments are rejected. Derived first-match clients return the first position or −1. Counts include every executed code-unit equality test, including mismatches.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

foreach(var pair in new[]{("aaaaa","aaa"),("abracadabra","cad"),("abc",""),("","x")})
{
 var result=PatternScan.Direct(pair.Item1,pair.Item2);
 Console.WriteLine($"text=[{pair.Item1}], pattern=[{pair.Item2}], positions=[{string.Join(",",result.Positions)}], comparisons={result.Comparisons}");
}
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


The loop tests each legal start once. At offset j, the first j code units match; a mismatch rules out that start, while j=m proves an occurrence. For n text units and m>0 pattern units, worst-case comparison work is O((n−m+1)m) when m≤n, commonly written O(nm); m>n gives zero tests. Returned r positions cost O(r) storage. Empty patterns require O(n) output despite zero equality tests.

## Proper Prefixes Survive a Mismatch

```{index} Proper Prefixes Survive a Mismatch
```

prefix[i] is the longest proper prefix of pattern[0..i] that is also its suffix. Proper excludes the whole segment. For ababaca, the table is 0,0,1,2,3,0,1. If q units matched and the next comparison fails, only borders of the matched prefix can remain possible: set q=prefix[q−1] and try again.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var matcher=new PatternScan("ababaca");
Console.WriteLine("prefix="+string.Join(",",matcher.Prefix));
Console.WriteLine($"preparation comparisons={matcher.PreparationComparisons}");
var snapshot=matcher.Prefix;snapshot[2]=99;
Console.WriteLine("owned prefix="+string.Join(",",matcher.Prefix));
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


Prefix construction maintains a valid border length for the already processed prefix. On mismatch, following the previous border chain finds the next possible border rather than resetting blindly or skipping a valid candidate. Successful comparisons increase q by at most one; fallback decreases it. Across all positions the total fallback work is linear in m. The owned table is prepared once and can be reused across multiple texts.

## KMP Keeps the Text Moving Forward

```{index} KMP Keeps the Text Moving Forward
```

Before each comparison, q code units of the pattern equal the suffix of text already consumed. After a mismatch, fallback preserves that invariant without moving the text index backward. After a complete match, fall back to the pattern’s longest border so overlapping occurrences remain discoverable.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var matcher=new PatternScan("aaa");
foreach(string text in new[]{"aaaaa","baaab",""})
{
 var result=matcher.Search(text);
 Console.WriteLine($"text=[{text}], positions=[{string.Join(",",result.Positions)}], scan comparisons={result.Comparisons}");
}
Console.WriteLine("empty boundaries="+string.Join(",",new PatternScan("").Search("abc").Positions));
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


Both preprocessing and scan use one counted equality test per candidate attempt, avoiding a hidden duplicate comparison in a while condition followed by an if. Preprocessing is O(m), each scan O(n), and the table O(m) space plus O(r) result storage. Total fallback decreases are bounded by total increases in matched length, which gives the linear bound despite the nested loop. Preparing for each request loses the benefit of repeated-pattern reuse.

## Measure a Complete Workload and Define Text Units

```{index} Measure a Complete Workload and Define Text Units
```

On repetitive text, direct scanning can repeatedly compare long matching prefixes. Include preparation in the KMP total, and reuse the same prepared pattern when comparing several texts. Operation counts are evidence, not timings.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

string text=new string('a',200)+"b",pattern=new string('a',20)+"b";
var direct=PatternScan.Direct(text,pattern);var matcher=new PatternScan(pattern);var kmp=matcher.Search(text);
if(!direct.Positions.SequenceEqual(kmp.Positions))throw new Exception("Matcher disagreement.");
Console.WriteLine($"positions={string.Join(",",kmp.Positions)}, direct={direct.Comparisons}, KMP prep={matcher.PreparationComparisons}, scan={kmp.Comparisons}, total={matcher.PreparationComparisons+kmp.Comparisons}");
string unicode="A😀B";
Console.WriteLine($"UTF-16 length={unicode.Length}, B index={new PatternScan("B").Search(unicode).Positions[0]}");
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


UTF-16 offsets count code units, not user-perceived characters; a supplementary emoji takes two units. Exact ordinal matching performs no case folding, culture rules, or Unicode normalization. Canonically equivalent spellings can therefore differ. For ordinary application code, use .NET search APIs with an explicit comparison policy.[^1] Do not claim this educational implementation outperforms the library without a controlled benchmark.

## Exercise

```{index} Exercise
```

Compare direct and KMP outputs for overlapping, empty, longer-than-text, and case-sensitive patterns. Check against ordinal library first-match results.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var cases=new[]{("aaaa","aa"),("abc",""),("ab","abcd"),("Audit","audit")};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

// Solution
var cases=new[]{("aaaa","aa"),("abc",""),("ab","abcd"),("Audit","audit")};
foreach(var pair in cases)
{
 var direct=PatternScan.Direct(pair.Item1,pair.Item2);var kmp=new PatternScan(pair.Item2).Search(pair.Item1);
 int first=kmp.Positions.Length==0?-1:kmp.Positions[0];
 if(!direct.Positions.SequenceEqual(kmp.Positions)||first!=pair.Item1.IndexOf(pair.Item2,StringComparison.Ordinal))throw new Exception("Ordinal mismatch.");
 Console.WriteLine($"[{pair.Item1}]/[{pair.Item2}]: positions=[{string.Join(",",kmp.Positions)}], first={first}");
}
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


[aaaa]/[aa]: positions=[0,1,2], first=0
[abc]/[]: positions=[0,1,2,3], first=0
[ab]/[abcd]: positions=[], first=-1
[Audit]/[audit]: positions=[], first=-1


```{rubric} Footnotes
```
[^1]: Microsoft documents the comparison overloads and empty-pattern behavior of [String.IndexOf](https://learn.microsoft.com/en-us/dotnet/api/system.string.indexof) and recommends making comparison choices explicit in [string best practices](https://learn.microsoft.com/en-us/dotnet/standard/base-types/best-practices-strings).
[^2]: KMP’s linear bound concerns code-unit comparisons and pattern preparation; returning every match also requires output storage.